# 14 · Introducción a los modelos de difusión

**Objetivo:** visualizar ruido, denoising iterativo, seed, steps y scheduler sin descargar un modelo grande.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
alto, ancho = 128, 128
y, x = np.ogrid[:alto, :ancho]
objetivo = np.exp(-((x - 64)**2 + (y - 64)**2) / (2 * 22**2))
plt.imshow(objetivo, cmap="gray", vmin=0, vmax=1)
plt.title("Imagen objetivo")
plt.axis("off")
plt.show()

## 📖 Forward process: agregar ruido

Durante el entrenamiento se crean versiones cada vez más ruidosas de una imagen.

In [ ]:
niveles = [0.0, 0.25, 0.5, 0.75, 1.0]
fig, axes = plt.subplots(1, len(niveles), figsize=(15, 3))
for ax, nivel in zip(axes, niveles):
    ruido = rng.normal(0, 1, objetivo.shape)
    mezcla = np.sqrt(1 - nivel) * objetivo + np.sqrt(nivel) * ruido
    ax.imshow(mezcla, cmap="gray")
    ax.set_title(f"ruido={nivel:.2f}")
    ax.axis("off")
plt.show()

## 📖 Reverse process: retirar ruido

Un modelo real predice el ruido en cada paso. Aquí simularemos el proceso acercándonos gradualmente a la imagen objetivo.

In [ ]:
seed = 123
steps = 8
rng = np.random.default_rng(seed)
actual = rng.normal(0, 1, objetivo.shape)
historial = [actual.copy()]
for paso in range(steps):
    fuerza = 0.25
    actual = actual + fuerza * (objetivo - actual)
    historial.append(actual.copy())
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
indices = np.linspace(0, steps, 5, dtype=int)
for ax, indice in zip(axes, indices):
    ax.imshow(historial[indice], cmap="gray")
    ax.set_title(f"paso {indice}")
    ax.axis("off")
plt.show()

## 🧪 Experimentos

- Cambia `seed`: cambia el ruido inicial.
- Cambia `steps`: cambia la cantidad de iteraciones.
- Cambia `fuerza`: simula un scheduler más agresivo o suave.

In [ ]:
def generar_simulacion(seed, steps=10, fuerza=0.22):
    rng = np.random.default_rng(seed)
    actual = rng.normal(0, 1, objetivo.shape)
    for _ in range(steps):
        actual += fuerza * (objetivo - actual)
    return actual

a = generar_simulacion(seed=10)
b = generar_simulacion(seed=11)
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(a, cmap="gray")
axes[0].set_title("Seed 10")
axes[1].imshow(b, cmap="gray")
axes[1].set_title("Seed 11")
for ax in axes:
    ax.axis("off")
plt.show()

## 🔗 Equivalencia aproximada con ComfyUI

- Ruido inicial → `RandomNoise` / seed.
- Pasos iterativos → sampler steps.
- Forma de avanzar → scheduler.
- Predicción de ruido → diffusion model / UNet / DiT.
- Imagen comprimida → latent.
- Reconstrucción final → VAE Decode.

## ✅ Cierre de la primera etapa

Ya tienes la base para continuar con autoencoders, VAE, embeddings de texto, Diffusers, image-to-image, inpainting y LoRA.